# 01 — Data Exploration: California Wildfires

**Goal:** Load the wildfire dataset and get to know it *before* cleaning or analyzing anything.

**Dataset:** Fire Program Analysis Fire-Occurrence Database (FPA FOD), 6th Edition, USDA Forest Service.
Short, Karen C. 2022. *Spatial wildfire occurrence data for the United States, 1992-2020* [FPA_FOD_20221014].
https://doi.org/10.2737/RDS-2013-0009.6

The data comes as a SQLite database file (`data/raw/FPA_FOD_20221014.sqlite`). It covers the whole United States,
so we will only load the California rows.

In this notebook we will look at:
1. The first few rows
2. The shape (rows × columns)
3. Column names and data types
4. Missing values
5. Basic statistics
6. Obvious data-quality problems

## 1. Imports

- **pandas** (`pd`) — works with tables of data (called *DataFrames*).
- **NumPy** (`np`) — fast math on numbers; pandas is built on top of it.
- **sqlite3** — built into Python; lets us open the `.sqlite` database file.

In [ ]:
import sqlite3

import numpy as np
import pandas as pd

## 2. Load the California wildfire records

A SQLite database can hold several tables. The wildfire records are in a table called `Fires`.

We use a SQL query to ask for only the columns we need and only rows where `STATE = 'CA'`.
We skip the `Shape` column: it stores the map point in a binary format we don't need, because
`LATITUDE` and `LONGITUDE` already give us the location.

The file path starts with `../` because this notebook lives inside the `notebooks/` folder,
so `..` means "go up one folder" to the project root.

In [ ]:
DATABASE_PATH = "../data/raw/FPA_FOD_20221014.sqlite"

query = """
SELECT
    FOD_ID,
    FIRE_NAME,
    FIRE_YEAR,
    DISCOVERY_DATE,
    DISCOVERY_DOY,
    CONT_DATE,
    NWCG_CAUSE_CLASSIFICATION,
    NWCG_GENERAL_CAUSE,
    FIRE_SIZE,
    FIRE_SIZE_CLASS,
    LATITUDE,
    LONGITUDE,
    OWNER_DESCR,
    STATE,
    COUNTY,
    FIPS_NAME
FROM Fires
WHERE STATE = 'CA'
"""

connection = sqlite3.connect(DATABASE_PATH)
wildfires = pd.read_sql_query(query, connection)
connection.close()

## 3. First look

`head()` shows the first 5 rows. It's the quickest way to see what the data actually looks like.

In [ ]:
wildfires.head()

## 4. Shape

`shape` returns `(number of rows, number of columns)`. Each row is one wildfire.

In [ ]:
wildfires.shape

## 5. Column names and data types

- `columns` lists every column name.
- `info()` shows each column's data type and how many non-missing values it has.

Pay attention to the data types. For example, is `DISCOVERY_DATE` stored as a real date, or as text?

In [ ]:
wildfires.columns

In [ ]:
wildfires.info()

## 6. What do the columns mean?

The dataset comes with a file describing every variable. Let's read it instead of guessing.

In [ ]:
variable_descriptions = pd.read_csv("../data/raw/_variable_descriptions.csv")
variable_descriptions

## 7. Missing values

`isna()` marks each missing cell as `True`; `.sum()` then counts them per column.
Dividing by the number of rows gives the **percentage** missing, which is easier to judge.

In [ ]:
missing_counts = wildfires.isna().sum()
missing_percent = (missing_counts / len(wildfires) * 100).round(1)

missing_summary = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_percent": missing_percent,
})
missing_summary.sort_values("missing_percent", ascending=False)

**Write down:** Which columns have the most missing values? Does it matter for our research question?
(Hint: we can still locate a fire using latitude/longitude even if its county is missing.)

## 8. Descriptive statistics

`describe()` gives count, mean, standard deviation, min, quartiles, and max for numeric columns.

Look closely at `FIRE_SIZE` (acres). Compare the **mean** with the **median** (the `50%` row).
If the mean is much larger than the median, a small number of huge fires are pulling the average up.
That's called a **right-skewed** distribution.

In [ ]:
wildfires.describe()

In [ ]:
print("Mean fire size (acres):  ", wildfires["FIRE_SIZE"].mean())
print("Median fire size (acres):", wildfires["FIRE_SIZE"].median())

## 9. Fire size classes

`FIRE_SIZE_CLASS` groups fires by size. According to the NWCG standard:

| Class | Acres |
|---|---|
| A | 0 – 0.25 |
| B | 0.26 – 9.9 |
| C | 10 – 99.9 |
| D | 100 – 299 |
| E | 300 – 999 |
| F | 1,000 – 4,999 |
| G | 5,000+ |

`value_counts()` counts how many rows fall into each category.

In [ ]:
wildfires["FIRE_SIZE_CLASS"].value_counts().sort_index()

## 10. Other quick checks

**Years covered** and **fire causes**:

In [ ]:
print("Earliest year:", wildfires["FIRE_YEAR"].min())
print("Latest year:  ", wildfires["FIRE_YEAR"].max())

In [ ]:
wildfires["NWCG_GENERAL_CAUSE"].value_counts()

**Latitude/longitude sanity check.** California spans roughly latitude 32.5° to 42° N and
longitude −124.5° to −114° W. Any points far outside that box would be data errors.

In [ ]:
outside_california = wildfires[
    (wildfires["LATITUDE"] < 32) | (wildfires["LATITUDE"] > 42.1) |
    (wildfires["LONGITUDE"] < -124.6) | (wildfires["LONGITUDE"] > -114)
]
print("Records outside California's rough bounding box:", len(outside_california))

**Duplicate check.** `FOD_ID` is the unique ID for each fire record, so it should never repeat.

In [ ]:
print("Duplicate FOD_ID values:", wildfires["FOD_ID"].duplicated().sum())
print("Fully duplicated rows:  ", wildfires.duplicated().sum())

**How are the dates stored?**

In [ ]:
print(wildfires["DISCOVERY_DATE"].head())
print("Data type:", wildfires["DISCOVERY_DATE"].dtype)

## 11. Data-quality notes (fill in after running)

Replace each bullet with what **you** observed. These notes drive the cleaning notebook.

- **Missing values:** _which columns, how many, and does it matter?_
- **Dates:** _what format are they in? what needs to change?_
- **Fire size:** _mean vs. median, and what that says about skew_
- **Size classes:** _what share of fires are tiny (class A/B)?_
- **Locations:** _any points outside California?_
- **Duplicates:** _any?_
- **Other surprises:**